# M3 Hadoop MapReduce — Full Runnable Version
This has all code included and runs without Hadoop Streaming.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
M3 Hadoop MapReduce Practice — Instructor/Solution Version
Safe local MapReduce version for VS Code and Jupyter. No Hadoop Streaming hang.
"""
from pathlib import Path
import subprocess
import sys
import shutil

BASE_DIR = Path("/Users/yuzhang/projects/UCI_Hadoop_Apache_Airflow/Assignments")
CODEBOOK_DIR = BASE_DIR / "Codebook"
DATASET_DIR = BASE_DIR / "Dataset"
OUTPUT_DIR = BASE_DIR / "Outputs" / "student_mapreduce_output"
INPUT_FILE = DATASET_DIR / "student_reviews.txt"
MAPPER = CODEBOOK_DIR / "mapper2.py"
REDUCER = CODEBOOK_DIR / "reducer.py"


def ensure_folders():
    CODEBOOK_DIR.mkdir(parents=True, exist_ok=True)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Codebook folder: {CODEBOOK_DIR}")
    print(f"[INFO] Dataset folder:  {DATASET_DIR}")
    print(f"[INFO] Output folder:   {OUTPUT_DIR}")


def ensure_sample_input():
    if not INPUT_FILE.exists():
        INPUT_FILE.write_text(
            """Hadoop makes big data processing possible.
"
            "MapReduce uses a mapper and a reducer.
"
            "Python can run mapper reducer logic locally.
"
            "Students practice Hadoop MapReduce with Python.
"
            "Hadoop Hadoop Spark Python Python MapReduce.
""",
            encoding="utf-8"
        )
        print(f"[INFO] Created sample input file: {INPUT_FILE}")
    else:
        print(f"[INFO] Input file already exists: {INPUT_FILE}")


def ensure_mapper_reducer_exist():
    missing = []
    if not MAPPER.exists():
        missing.append(str(MAPPER))
    if not REDUCER.exists():
        missing.append(str(REDUCER))
    if missing:
        raise FileNotFoundError(
            "Missing required file(s):
" + "
".join(missing) +
            "

Please place mapper2.py and reducer.py in the Codebook folder."
        )
    print(f"[INFO] Mapper found:  {MAPPER}")
    print(f"[INFO] Reducer found: {REDUCER}")


def clean_output():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
        print("[INFO] Previous output removed.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def run_mapper():
    input_text = INPUT_FILE.read_text(encoding="utf-8", errors="ignore")
    mapper_process = subprocess.run(
        [sys.executable, str(MAPPER)],
        input=input_text,
        text=True,
        capture_output=True,
        check=True
    )
    mapped_lines = mapper_process.stdout.splitlines()
    print(f"[INFO] Mapper produced {len(mapped_lines)} key-value rows.")
    return mapped_lines


def sort_mapper_output(mapped_lines):
    sorted_lines = sorted(line for line in mapped_lines if line.strip())
    print(f"[INFO] Sorted {len(sorted_lines)} rows before reducer.")
    return sorted_lines


def run_reducer(sorted_lines):
    reducer_input = "
".join(sorted_lines) + "
"
    reducer_process = subprocess.run(
        [sys.executable, str(REDUCER)],
        input=reducer_input,
        text=True,
        capture_output=True,
        check=True
    )
    reduced_text = reducer_process.stdout
    print("[INFO] Reducer completed.")
    return reduced_text


def save_output(reduced_text):
    output_file = OUTPUT_DIR / "part-00000"
    output_file.write_text(reduced_text, encoding="utf-8")
    print(f"[INFO] Output saved to: {output_file}")
    return output_file


def display_output(output_file):
    print("
========== FINAL WORD COUNT OUTPUT ==========")
    print(output_file.read_text(encoding="utf-8"))


def main():
    ensure_folders()
    ensure_sample_input()
    ensure_mapper_reducer_exist()
    clean_output()
    mapped_lines = run_mapper()
    sorted_lines = sort_mapper_output(mapped_lines)
    reduced_text = run_reducer(sorted_lines)
    output_file = save_output(reduced_text)
    display_output(output_file)


if __name__ == "__main__":
    main()